In [7]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
JupyterDash.infer_jupyter_proxy_config()

from CRUD_Python_Module import AnimalShelter

# aacuser was created in admin during Module 3, so authSource stays in the class
username = "aacuser"
password = "<your password>"
shelter = AnimalShelter(username, password)

# Empty query is the unfiltered load. Drop _id or the DataTable cannot render ObjectId.
df = pd.DataFrame.from_records(shelter.read({}))
df.drop(columns=['_id'], inplace=True)

# Logo is already in code_files. Spec requires the image to link to the client home page.
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app = JupyterDash(__name__)

app.layout = html.Div([
    html.Center([
        html.A(
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height': '150px'}),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.B(html.H1('SNHU CS-340 Dashboard - Darius Tatum'))
    ]),
    html.Hr(),
    # Four required filters. Reset is the default so the first view is unfiltered.
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '15px'}
        )
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable='single',
        selected_rows=[0],
        page_size=10,
        page_action='native',
        sort_action='native',
        filter_action='native'
    ),
    html.Br(),
    html.Hr(),
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ]),
    html.H3('Darius Tatum')
])

def rescue_query(filter_type):
    # Breeds, sex, and age come from the Rescue Type table in the specs.
    # Regex covers "Mix" and combined breeds. Age is in weeks, not the text field.
    if filter_type == 'water':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {"$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland"}
        }
    if filter_type == 'mountain':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {"$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler"}
        }
    if filter_type == 'disaster':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "breed": {"$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler"}
        }
    return {}

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Query goes through the CRUD class, not a direct MongoClient call.
    records = shelter.read(rescue_query(filter_type))
    table = pd.DataFrame.from_records(records)
    if '_id' in table.columns:
        table.drop(columns=['_id'], inplace=True)
    return table.to_dict('records')

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    # Pie chart is the second required chart. It follows the table, so a filter updates it.
    if viewData is None or len(viewData) == 0:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    return [dcc.Graph(figure=px.pie(dff, names='breed', title='Breeds'))]

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    # Names instead of column numbers. The blank CSV column shifts iloc.
    if viewData is None or len(viewData) == 0: 
        return []
    dff = pd.DataFrame.from_dict(viewData)
    row = 0 if index is None or len(index) == 0 else index[0]
    if row >= len(dff):
        row = 0
    return [
        dl.Map(style={'width': '500px', 'height': '400px'},
               center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id='base-layer-id'),
            dl.Marker(position=[float(dff.iloc[row]['location_lat']),
                                float(dff.iloc[row]['location_long'])], children=[
                dl.Tooltip(dff.iloc[row]['breed']),
                dl.Popup([
                    html.H1('Animal Name'),
                    html.P(dff.iloc[row]['name'])
                ])
            ])
        ])
    ]

app.run_server()

Dash app running on https://hermanigor-britishproton-3000.codio.io/proxy/8050/
